# Energy-Aware Scheduling Benchmark

Compares the **probabilistic** (energy-aware) scheduler against the **round-robin** baseline across five workloads: three lightweight (addition, naive-fib, matrix-mul) and two compute-heavy variants (~5 s each) that give the scheduler more opportunity to demonstrate energy savings.

Each task records five timestamps used throughout this notebook to decompose end-to-end latency into phases:

| Timestamp | Phase boundary |
|---|---|
| `start_time` → `scheduler_time` | Scheduling overhead |
| `scheduler_time` → `proplet_arrive_time` | Queue / network transfer |
| `proplet_arrive_time` → `execution_time` | Wait for executor |
| `execution_time` → `finish_time` | Wasm execution |

# Setup & Helper functions

In [ ]:
import os
# When the notebook lives in notebooks/, set cwd to the project root so that
# relative paths (build/, benchmark-results/, ...) resolve correctly.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import json
import requests
import scipy
import time
import dateutil
import matplotlib.pyplot as plt
import matplotlib
import concurrent
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED, as_completed

In [ ]:
SCHEDULERS = ["roundrobin", "probabilistic", "static"]
TASKS_URL = "http://localhost:7070/tasks"
THROUGHPUT_CLIENTS = [1, 2, 4, 8]
THROUGHPUT_DURATION = 30  # seconds
REP_COUNTS = [500]  # default; overridden per workload via "rep_counts"
WORKLOADS = {
    "addition": {
        "function_name": "add",
        "path": "addition.wasm",
        "inputs": [10, 20],
    },
    "naive-fib": {
        "function_name": "naive_fib",
        "path": "naive-fib.wasm",
        "inputs": [40],
    },
    "naive-fib-heavy": {
        "function_name": "naive_fib",
        "path": "naive-fib.wasm",
        "inputs": [44],          # ~5s per task
        "rep_counts": [50],
        "skip_warmup": True,     # 50 x 5s tasks would exceed warmup budget
        "skip_throughput": True, # 30s window yields too few completions for TPS stats
    },
    "matrix-mul": {
        "function_name": "matrix_mul",
        "path": "matrix-mul.wasm",
        "inputs": [40],
    },
    "matrix-mul-heavy": {
        "function_name": "matrix_mul",
        "path": "matrix-mul.wasm",
        "inputs": [500],         # ~5s per task
        "rep_counts": [50],
        "skip_warmup": True,
        "skip_throughput": True,
    },
}

In [ ]:
def create_task(task_name, task_inputs, scheduler = None) -> str:
    body = {"name": task_name, "inputs": task_inputs}
    if (scheduler):
        body["scheduler"] = scheduler
    resp = requests.post(
        TASKS_URL,
        headers={"Content-Type": "application/json"},
        data=json.dumps(body),
    )
    resp.raise_for_status()
    task_id = resp.json()["id"]
    return task_id

def upload_wasm(task_id, task_location):
    with open(task_location, "rb") as f:
        upload = requests.put(
            f"{TASKS_URL}/{task_id}/upload",
            files={"file": f},
        )
        upload.raise_for_status()

def prepare_tasks(task_count, task_name, task_location, task_inputs, scheduler = None):
    task_ids = []
    for i in range(task_count):
        print(f"Creating task {i+1}/{task_count}...")
        task_id = create_task(task_name, task_inputs, scheduler)
        upload_wasm(task_id, task_location)
        task_ids.append(task_id)
    return task_ids

In [ ]:
def start_task(task_id):
    resp = requests.post(f"{TASKS_URL}/{task_id}/start")
    resp.raise_for_status()

def start_tasks(task_ids):
    for task_id in task_ids:
        start_task(task_id)

In [ ]:
def get_task_result(task_id):
    while True:
        resp = requests.get(f"{TASKS_URL}/{task_id}")
        resp.raise_for_status()
        data = resp.json()
        if "results" in data:
            return data
            break
        time.sleep(0.005)

def get_tasks_results(task_ids):
    results = []
    for task_id in task_ids:
        data = get_task_result(task_id)
        results.append(data)
    return results

In [ ]:
def execute_benchmark(function_name, task_location, task_inputs, task_count = 1, scheduler = None):
    print(f"Starting test execution for task: {function_name}")

    task_ids = prepare_tasks(task_count, function_name, task_location, task_inputs, scheduler)
    print("Prepared all tasks successfully")

    start_execution = time.perf_counter_ns()

    # Start all tasks before polling any result, so they run concurrently on the proplets.
    start_tasks(task_ids)
    print("Started all tasks successfully")

    results = get_tasks_results(task_ids)

    duration = (time.perf_counter_ns() - start_execution) * 1000 # ms
    print(f"Retrieved all task results successfully, start to finish took {duration} ms")

    return results, duration

In [ ]:
def _submit_task_start(workload, input_data, scheduler, task_location):
    task_id = create_task(workload, input_data, scheduler)
    upload_wasm(task_id, task_location)
    start_task(task_id)
    return task_id

def _to_unix_seconds(ts):
    """Return a Unix timestamp in seconds, or None if the value is absent/unset.

    Go serialises time.Time zero values as "0001-01-01T00:00:00Z". These are
    fields that the runtime never populated (e.g. proplet_arrive_time before a
    task reaches a proplet). Treating them as None lets all callers skip the
    phase safely via a simple truthiness check.
    """
    if ts is None:
        return None
    if isinstance(ts, (int, float)):
        return float(ts) if ts > 0 else None  # 0 = Unix epoch, also not a real task time
    parsed = dateutil.parser.isoparse(ts)
    if parsed.year < 2:   # Go zero time is year 0001
        return None
    return parsed.timestamp()

In [ ]:
def throughput_benchmark(scheduler, workload, input_data, task_location, concurrency, duration_sec=30):
    submitted_task_ids = []
    submit_start = time.time()
    submit_deadline = submit_start + duration_sec

    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        in_flight = set()

        while time.time() < submit_deadline:
            while len(in_flight) < concurrency and time.time() < submit_deadline:
                fut = executor.submit(_submit_task_start, workload, input_data, scheduler, task_location)
                in_flight.add(fut)

            done, _ = wait(in_flight, timeout=0.0, return_when=FIRST_COMPLETED)
            for fut in done:
                in_flight.remove(fut)
                try:
                    task_id = fut.result()
                    submitted_task_ids.append(task_id)
                except Exception:
                    pass

        for fut in as_completed(in_flight):
            try:
                task_id = fut.result()
                submitted_task_ids.append(task_id)
            except Exception:
                pass

    # Record the cutoff immediately after the submission window closes. Tasks that finish
    # after this point are excluded from TPS — they benefited from extra time beyond the window.
    cutoff_unix = time.time()

    finished_before_cutoff = []
    for task_id in submitted_task_ids:
        try:
            data = get_task_result(task_id)
            finish_unix = _to_unix_seconds(data.get("finish_time"))
            if finish_unix is not None and finish_unix <= cutoff_unix:
                finished_before_cutoff.append(data)
        except Exception:
            pass

    throughput = len(finished_before_cutoff) / duration_sec

    return {
        "submitted": len(submitted_task_ids),
        "completed_before_cutoff": len(finished_before_cutoff),
        "throughput_tps": throughput,
        "cutoff_unix": cutoff_unix,
        "results_before_cutoff": finished_before_cutoff,
    }

# Benchmark Execution

## Warmup

Runs tasks for 180 s before measuring. This brings the system to a thermal steady state so that measured energy and latency values reflect stable operating conditions rather than cold-start behaviour. Heavy workloads are excluded to stay within the time budget.

In [ ]:
WARMUP_BATCH_SIZE = 50
WARMUP_DURATION = 180  # seconds — enough time for thermal plateau

print("Warming up system...")
warmup_deadline = time.time() + WARMUP_DURATION
iteration = 0
while time.time() < warmup_deadline:
    for task_name, task_info in WORKLOADS.items():
        if task_info.get("skip_warmup"):
            continue
        execute_benchmark(
            function_name=task_info["function_name"],
            task_count=WARMUP_BATCH_SIZE,
            task_location=f"build/{task_info['path']}",
            task_inputs=task_info["inputs"],
            scheduler=SCHEDULERS[0],
        )
        if time.time() >= warmup_deadline:
            break
    iteration += 1
    print(f"  Warmup iteration {iteration} done, {max(0, warmup_deadline - time.time()):.0f}s remaining")
print("Warmup complete.")

## Latency

Submits `rep_count` tasks per scheduler/workload combination (500 for lightweight workloads, 50 for heavy ones), starts them all, then polls for results. The raw task records — including all five timestamps and `energy_consumed` — are stored in `results` and analysed in the Results section below.

In [ ]:
results = {}

for scheduler in SCHEDULERS:
    for task_name, task_info in WORKLOADS.items():
        for rep_count in task_info.get("rep_counts", REP_COUNTS):
            print(f"Running benchmark for scheduler: {scheduler}, task: {task_name}, rep count: {rep_count}")
            task_results, duration = execute_benchmark(
                function_name=task_info["function_name"],
                task_count=rep_count,
                task_location=f"build/{task_info['path']}",
                task_inputs=task_info["inputs"],
                scheduler=scheduler,
            )
            results[f"{scheduler}_{task_name}_{rep_count}"] = {
                "results": task_results,
                "duration": duration,
            }

## Throughput

Submits tasks continuously at each concurrency level for `THROUGHPUT_DURATION` seconds. TPS is computed as tasks completed before the submission window closed divided by the window length. Heavy workloads are excluded — at ~5 s per task, a 30 s window yields too few completions for a meaningful TPS figure.

In [ ]:
throughput_benchmark_results = {}

for scheduler in SCHEDULERS:
    print(f"Running throughput benchmark for scheduler: {scheduler}")
    for task_name, task_info in WORKLOADS.items():
        if task_info.get("skip_throughput"):
            continue
        print(f"Running throughput benchmark for task: {task_name}")
        for concurrency in THROUGHPUT_CLIENTS:
            result = throughput_benchmark(
                scheduler=scheduler,
                workload=task_info["function_name"],
                input_data=task_info["inputs"],
                task_location=f"build/{task_info['path']}",
                concurrency=concurrency,
                duration_sec=THROUGHPUT_DURATION,
            )
            throughput_benchmark_results[f"{scheduler}_{task_name}_{concurrency}"] = {
                "scheduler": scheduler,
                "task": task_name,
                "concurrency": concurrency,
                "submitted": result["submitted"],
                "completed_before_cutoff": result["completed_before_cutoff"],
                "throughput_tps": result["throughput_tps"],
            }

# Results

Run these cells after the benchmark execution above has completed. Charts are saved to `benchmark-results/`; numerical statistics are printed inline and exported to CSV at the end.

In [48]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import os
os.makedirs("benchmark-results", exist_ok=True)


for task_name, task_info in WORKLOADS.items():
    for rep_count in task_info.get("rep_counts", REP_COUNTS):
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

        sched_proplet_counts = {}
        sched_delays = {}
        sched_energy = {}

        for scheduler in SCHEDULERS:
            proplet_counts = {}
            proplet_delays = {}
            energy_values = []

            value = results.get(f"{scheduler}_{task_name}_{rep_count}")
            if value:
                for task_result in value["results"]:
                    pid = task_result["proplet_id"]
                    proplet_counts[pid] = proplet_counts.get(pid, 0) + 1
                    start_ts = _to_unix_seconds(task_result.get("start_time"))
                    finish_ts = _to_unix_seconds(task_result.get("finish_time"))
                    if start_ts is not None and finish_ts is not None:
                        proplet_delays.setdefault(pid, []).append((finish_ts - start_ts) * 1000)

                    energy = task_result.get("energy_consumed")
                    if energy is not None:
                        energy_values.append(energy)

            label_map = {pid: f"Proplet {i+1}" for i, pid in enumerate(proplet_counts)}
            sched_proplet_counts[scheduler] = {label_map[pid]: c for pid, c in proplet_counts.items()}
            sched_delays[scheduler] = {label_map[pid]: d for pid, d in proplet_delays.items()}
            sched_energy[scheduler] = energy_values

        # ---- Tasks per proplet (grouped by scheduler) ----
        all_proplets = sorted({label for d in sched_proplet_counts.values() for label in d.keys()})
        x = range(len(SCHEDULERS))
        width = 0.8 / max(len(all_proplets), 1)

        for j, proplet_label in enumerate(all_proplets):
            counts = [sched_proplet_counts[s].get(proplet_label, 0) / sum(sched_proplet_counts[s].values()) if sum(sched_proplet_counts[s].values()) > 0 else 0 for s in SCHEDULERS]
            bars = ax1.bar([xi + j*width for xi in x], counts, width=width, label=proplet_label)
            for bar, count in zip(bars, counts):
                ax1.annotate(f"{count:.2f}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                             ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")

        ax1.set_xticks([xi + width*(len(all_proplets)-1)/2 for xi in x])
        ax1.set_xticklabels(SCHEDULERS)
        ax1.set_xlabel("Scheduler")
        ax1.set_ylabel("Fraction of tasks")
        ax1.set_title(f"Tasks per Proplet")
        ax1.legend(title="Proplet")

        # ---- Task delays (boxplots per scheduler) ----
        box_data = []
        box_labels = []
        for scheduler in SCHEDULERS:
            for label in all_proplets:
                box_data.append(sched_delays[scheduler].get(label, []))
                box_labels.append(f"{scheduler}\n{label}")

        ax2.boxplot(box_data, tick_labels=box_labels)
        ax2.set_xlabel("Scheduler / Proplet")
        ax2.set_ylabel("End-to-end latency (ms)")
        ax2.set_title(f"Task Latency")
        ax2.yaxis.grid(True)

        # ---- Per-task energy distribution (boxplot per scheduler) ----
        energy_box_data = [sched_energy[scheduler] for scheduler in SCHEDULERS]
        ax3.boxplot(energy_box_data, tick_labels=SCHEDULERS)
        ax3.set_xlabel("Scheduler")
        ax3.set_ylabel("Energy consumed (model-estimated)")
        ax3.set_title(f"Per-task Energy Distribution")
        ax3.yaxis.grid(True)

        # ---- Total energy consumed per scheduler ----
        total_energy = {s: sum(sched_energy[s]) for s in SCHEDULERS}
        bars = ax4.bar(SCHEDULERS, [total_energy[s] for s in SCHEDULERS])
        for bar, s in zip(bars, SCHEDULERS):
            ax4.annotate(f"{total_energy[s]:.1f}",
                         (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                         ha="center", va="bottom", fontsize=10, xytext=(0, 3), textcoords="offset points")
        ax4.set_xlabel("Scheduler")
        ax4.set_ylabel("Total energy consumed (model-estimated)")
        ax4.set_title(f"Total Energy")
        ax4.yaxis.grid(True)

        fig.suptitle(f"{task_name} | n={rep_count}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"benchmark-results/results_{task_name}_{rep_count}.png", dpi=100, bbox_inches="tight")
        plt.close(fig)

## Overview Charts

One figure per workload with four panels: task distribution across proplets, end-to-end latency boxplots per proplet, per-task energy distribution, and total energy. Saved to `benchmark-results/results_<workload>_<n>.png`.

In [49]:
from scipy import stats as scipy_stats
import numpy as np

print("=== Energy Consumption Statistics ===\n")
for task_name, task_info in WORKLOADS.items():
    for rep_count in task_info.get("rep_counts", REP_COUNTS):
        print(f"Task: {task_name}, n={rep_count}")
        energy_by_sched = {}
        for scheduler in SCHEDULERS:
            value = results.get(f"{scheduler}_{task_name}_{rep_count}")
            if value:
                energy_by_sched[scheduler] = np.array(
                    [t["energy_consumed"] for t in value["results"] if t.get("energy_consumed") is not None],
                    dtype=float,
                )

        for scheduler, arr in energy_by_sched.items():
            ci = scipy_stats.t.interval(0.95, len(arr) - 1, loc=arr.mean(), scale=scipy_stats.sem(arr))
            print(f"  {scheduler}: mean={arr.mean():.3f}, std={arr.std(ddof=1):.3f}, "
                  f"p50={np.median(arr):.3f}, p95={np.percentile(arr, 95):.3f}, p99={np.percentile(arr, 99):.3f}, "
                  f"total={arr.sum():.1f}, CI95=[{ci[0]:.3f}, {ci[1]:.3f}], n={len(arr)}")

        if len(energy_by_sched) == 2:
            sched_a, sched_b = SCHEDULERS[0], SCHEDULERS[1]
            a, b = energy_by_sched[sched_a], energy_by_sched[sched_b]
            _, p = scipy_stats.mannwhitneyu(a, b, alternative="two-sided")
            comparisons = np.sign(a[:, None] - b[None, :])
            cliffs_d = float(comparisons.mean())
            mag = ("negligible" if abs(cliffs_d) < 0.147 else
                   "small"      if abs(cliffs_d) < 0.330 else
                   "medium"     if abs(cliffs_d) < 0.474 else "large")
            sig = "*" if p < 0.05 else ""
            print(f"  Mann-Whitney U: p={p:.4f}{sig}  Cliff's d={cliffs_d:+.3f} ({mag})")
            rr_total = energy_by_sched.get("roundrobin", np.array([0])).sum()
            prob_total = energy_by_sched.get("probabilistic", np.array([0])).sum()
            if rr_total > 0:
                savings_pct = (rr_total - prob_total) / rr_total * 100
                print(f"  Total energy savings vs roundrobin: {savings_pct:.2f}%")
        print()

=== Energy Consumption Statistics ===

Task: addition, n=500
  roundrobin: mean=2.234, std=0.916, p50=2.044, p95=3.655, p99=4.583, total=1117.2, CI95=[2.154, 2.315], n=500
  probabilistic: mean=2.093, std=0.650, p50=2.003, p95=3.312, p99=4.236, total=1046.3, CI95=[2.036, 2.150], n=500
  Mann-Whitney U: p=0.0432*  Cliff's d=+0.074 (negligible)
  Total energy savings vs roundrobin: 6.34%

Task: naive-fib, n=500
  roundrobin: mean=11225.246, std=2616.479, p50=10653.878, p95=15107.256, p99=15400.724, total=5612622.9, CI95=[10995.348, 11455.144], n=500
  probabilistic: mean=9751.046, std=2249.760, p50=9440.420, p95=13428.739, p99=14232.203, total=4875523.2, CI95=[9553.370, 9948.722], n=500
  Mann-Whitney U: p=0.0000*  Cliff's d=+0.392 (medium)
  Total energy savings vs roundrobin: 13.13%

Task: naive-fib-heavy, n=50
  roundrobin: mean=79946.988, std=17980.044, p50=75727.165, p95=104590.829, p99=104920.493, total=3997349.4, CI95=[74837.116, 85056.860], n=50
  probabilistic: mean=72640.180, s

## Energy Efficiency

Per-task energy statistics (mean, std, percentiles, 95 % CI) for each scheduler/workload combination. Statistical significance is tested with Mann-Whitney U (non-parametric); practical significance with Cliff's delta (|d| < 0.147 negligible, < 0.33 small, < 0.474 medium, ≥ 0.474 large).

In [50]:
import numpy as np
from scipy import stats as scipy_stats

def _phase_stats(arr):
    if len(arr) < 2:
        return {}
    a = np.array(arr, dtype=float)
    ci = scipy_stats.t.interval(0.95, len(a) - 1, loc=a.mean(), scale=scipy_stats.sem(a))
    return {
        "n": len(a),
        "mean":     float(a.mean()),
        "std":      float(a.std(ddof=1)),
        "median":   float(np.median(a)),
        "p95":      float(np.percentile(a, 95)),
        "p99":      float(np.percentile(a, 99)),
        "iqr":      float(np.percentile(a, 75) - np.percentile(a, 25)),
        "ci95_low":  float(ci[0]),
        "ci95_high": float(ci[1]),
    }

latency_stats = {}

print("=== Latency Phase Breakdown (ms) ===\n")
for task_name, task_info in WORKLOADS.items():
    for rep_count in task_info.get("rep_counts", REP_COUNTS):
        print(f"Task: {task_name}, n={rep_count}")
        phase_data_by_sched = {}

        for scheduler in SCHEDULERS:
            value = results.get(f"{scheduler}_{task_name}_{rep_count}")
            if not value:
                continue
            phases = {
                "e2e":                 [],
                "scheduling_overhead": [],
                "queue_transfer":      [],
                "wait_for_exec":       [],
                "wasm_execution":      [],
            }
            for t in value["results"]:
                start     = _to_unix_seconds(t.get("start_time"))
                sched     = _to_unix_seconds(t.get("scheduler_time"))
                arrive    = _to_unix_seconds(t.get("proplet_arrive_time"))
                exec_s    = _to_unix_seconds(t.get("execution_time"))
                finish    = _to_unix_seconds(t.get("finish_time"))
                if start and finish:
                    phases["e2e"].append((finish - start) * 1000)
                if start and sched:
                    phases["scheduling_overhead"].append((sched - start) * 1000)
                if sched and arrive:
                    phases["queue_transfer"].append((arrive - sched) * 1000)
                if arrive and exec_s:
                    phases["wait_for_exec"].append((exec_s - arrive) * 1000)
                if exec_s and finish:
                    phases["wasm_execution"].append((finish - exec_s) * 1000)

            phase_data_by_sched[scheduler] = phases
            latency_stats[f"{scheduler}_{task_name}_{rep_count}"] = {
                k: _phase_stats(v) for k, v in phases.items()
            }
            print(f"  {scheduler}:")
            for phase_name, vals in phases.items():
                s = _phase_stats(vals)
                if s:
                    print(f"    {phase_name:25s}: mean={s['mean']:8.3f}  std={s['std']:8.3f}  "
                          f"p50={s['median']:8.3f}  p95={s['p95']:8.3f}  p99={s['p99']:8.3f}  "
                          f"CI95=[{s['ci95_low']:.3f}, {s['ci95_high']:.3f}]")

        if len(phase_data_by_sched) == 2:
            sched_a, sched_b = SCHEDULERS[0], SCHEDULERS[1]
            print(f"  Statistical tests ({sched_a} vs {sched_b}):")
            for phase_name in ["e2e", "scheduling_overhead", "wasm_execution"]:
                a_vals = np.array(phase_data_by_sched[sched_a][phase_name], dtype=float)
                b_vals = np.array(phase_data_by_sched[sched_b][phase_name], dtype=float)
                if len(a_vals) < 2 or len(b_vals) < 2:
                    continue
                _, p = scipy_stats.mannwhitneyu(a_vals, b_vals, alternative="two-sided")
                comparisons = np.sign(a_vals[:, None] - b_vals[None, :])
                cliffs_d = float(comparisons.mean())
                mag = ("negligible" if abs(cliffs_d) < 0.147 else
                       "small"      if abs(cliffs_d) < 0.330 else
                       "medium"     if abs(cliffs_d) < 0.474 else "large")
                sig = "*" if p < 0.05 else ""
                print(f"    {phase_name:25s}: p={p:.4f}{sig:1s}  Cliff's d={cliffs_d:+.3f} ({mag})")
        print()

=== Latency Phase Breakdown (ms) ===

Task: addition, n=500
  roundrobin:
    e2e                      : mean= 333.092  std= 102.789  p50= 277.898  p95= 488.764  p99= 648.307  CI95=[324.060, 342.123]
    scheduling_overhead      : mean=   0.027  std=   0.058  p50=   0.019  p95=   0.037  p99=   0.217  CI95=[0.022, 0.032]
    queue_transfer           : mean=  60.622  std=  76.234  p50=  14.269  p95= 207.494  p99= 232.062  CI95=[53.924, 67.321]
  probabilistic:
    e2e                      : mean= 334.914  std= 103.603  p50= 291.397  p95= 477.564  p99= 641.714  CI95=[325.811, 344.018]
    scheduling_overhead      : mean=   0.022  std=   0.030  p50=   0.017  p95=   0.033  p99=   0.144  CI95=[0.020, 0.025]
    queue_transfer           : mean=  58.942  std=  75.593  p50=  13.761  p95= 206.024  p99= 228.086  CI95=[52.300, 65.584]
  Statistical tests (roundrobin vs probabilistic):
    e2e                      : p=0.5613   Cliff's d=+0.021 (negligible)
    scheduling_overhead      : p=0.0000*  

## Latency

Decomposes each task's end-to-end latency into the four phases defined in the introduction, then reports descriptive statistics and the same Mann-Whitney U + Cliff's delta tests for the three most relevant phases: e2e, scheduling overhead, and Wasm execution. A stacked bar chart follows.

In [51]:
print("=== Load Balance Quality ===\n")
for task_name, task_info in WORKLOADS.items():
    for rep_count in task_info.get("rep_counts", REP_COUNTS):
        print(f"Task: {task_name}, n={rep_count}")
        for scheduler in SCHEDULERS:
            value = results.get(f"{scheduler}_{task_name}_{rep_count}")
            if not value:
                continue
            counts = {}
            for t in value["results"]:
                pid = t.get("proplet_id", "unknown")
                counts[pid] = counts.get(pid, 0) + 1

            count_arr = np.array(list(counts.values()), dtype=float)
            n = len(count_arr)
            cv = count_arr.std(ddof=1) / count_arr.mean() if count_arr.mean() > 0 else float("nan")
            sorted_c = np.sort(count_arr)
            gini = (2.0 * np.dot(np.arange(1, n + 1), sorted_c) - (n + 1) * sorted_c.sum()) / (n * sorted_c.sum()) if sorted_c.sum() > 0 else float("nan")
            dist = {f"P{i+1}": int(v) for i, v in enumerate(count_arr)}
            total = int(count_arr.sum())
            fractions = {k: f"{v/total:.3f}" for k, v in dist.items()}
            print(f"  {scheduler}: proplets={n}, tasks={dist}, fractions={fractions}, CV={cv:.4f}, Gini={gini:.4f}")
        print()

=== Load Balance Quality ===

Task: addition, n=500
  roundrobin: proplets=3, tasks={'P1': 167, 'P2': 166, 'P3': 167}, fractions={'P1': '0.334', 'P2': '0.332', 'P3': '0.334'}, CV=0.0035, Gini=0.0013
  probabilistic: proplets=3, tasks={'P1': 209, 'P2': 185, 'P3': 106}, fractions={'P1': '0.418', 'P2': '0.370', 'P3': '0.212'}, CV=0.3234, Gini=0.1373

Task: naive-fib, n=500
  roundrobin: proplets=3, tasks={'P1': 166, 'P2': 168, 'P3': 166}, fractions={'P1': '0.332', 'P2': '0.336', 'P3': '0.332'}, CV=0.0069, Gini=0.0027
  probabilistic: proplets=3, tasks={'P1': 173, 'P2': 134, 'P3': 193}, fractions={'P1': '0.346', 'P2': '0.268', 'P3': '0.386'}, CV=0.1800, Gini=0.0787

Task: naive-fib-heavy, n=50
  roundrobin: proplets=3, tasks={'P1': 17, 'P2': 17, 'P3': 16}, fractions={'P1': '0.340', 'P2': '0.340', 'P3': '0.320'}, CV=0.0346, Gini=0.0133
  probabilistic: proplets=3, tasks={'P1': 19, 'P2': 20, 'P3': 11}, fractions={'P1': '0.380', 'P2': '0.400', 'P3': '0.220'}, CV=0.2960, Gini=0.1200

Task: mat

## Load Balance

Measures how evenly tasks are distributed across proplets. CV (coefficient of variation) and Gini coefficient both equal 0 for perfect equality; higher values indicate skew. This quantifies whether the probabilistic scheduler's energy routing produces unequal load distribution as a side effect.

In [52]:
phase_order  = ["scheduling_overhead", "queue_transfer", "wait_for_exec", "wasm_execution"]
phase_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
phase_labels = ["Scheduling overhead", "Queue / transfer", "Wait for executor", "Wasm execution"]

# Phases the scheduler directly controls — shown separately so they aren't
# dwarfed by queue/transfer in the full breakdown chart.
focus_phases  = ["scheduling_overhead", "wasm_execution"]
focus_colors  = ["#4C72B0", "#C44E52"]
focus_labels  = ["Scheduling overhead", "Wasm execution"]

for task_name, task_info in WORKLOADS.items():
    for rep_count in task_info.get("rep_counts", REP_COUNTS):
        x = np.arange(len(SCHEDULERS))
        width = 0.5

        # ---- Full breakdown ----
        fig, ax = plt.subplots(figsize=(7, 5))
        bottoms = np.zeros(len(SCHEDULERS))
        for color, phase, label in zip(phase_colors, phase_order, phase_labels):
            means = [
                latency_stats.get(f"{s}_{task_name}_{rep_count}", {}).get(phase, {}).get("mean", 0)
                for s in SCHEDULERS
            ]
            ax.bar(x, means, width, bottom=bottoms, label=label, color=color)
            bottoms += np.array(means)
        ax.set_xticks(x)
        ax.set_xticklabels(SCHEDULERS)
        ax.set_ylabel("Mean latency (ms)")
        ax.set_title(f"Latency Phase Breakdown | {task_name}, n={rep_count}")
        ax.legend(loc="upper right", fontsize=9)
        ax.yaxis.grid(True, zorder=0)
        plt.tight_layout()
        plt.savefig(f"benchmark-results/latency_phases_{task_name}_{rep_count}.png", dpi=100, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved latency_phases_{task_name}_{rep_count}.png")

        # ---- Scheduler-controlled phases only (zoomed) ----
        fig, ax = plt.subplots(figsize=(7, 5))
        bottoms = np.zeros(len(SCHEDULERS))
        any_data = False
        for color, phase, label in zip(focus_colors, focus_phases, focus_labels):
            means = [
                latency_stats.get(f"{s}_{task_name}_{rep_count}", {}).get(phase, {}).get("mean", 0)
                for s in SCHEDULERS
            ]
            if any(m > 0 for m in means):
                any_data = True
            ax.bar(x, means, width, bottom=bottoms, label=label, color=color)
            bottoms += np.array(means)
        ax.set_xticks(x)
        ax.set_xticklabels(SCHEDULERS)
        ax.set_ylabel("Mean latency (ms)")
        ax.set_title(f"Scheduler-Controlled Latency | {task_name}, n={rep_count}")
        ax.legend(loc="upper right", fontsize=9)
        ax.yaxis.grid(True, zorder=0)
        plt.tight_layout()
        if any_data:
            plt.savefig(f"benchmark-results/latency_phases_focus_{task_name}_{rep_count}.png", dpi=100, bbox_inches="tight")
            print(f"Saved latency_phases_focus_{task_name}_{rep_count}.png")
        plt.close(fig)

Saved latency_phases_addition_500.png
Saved latency_phases_focus_addition_500.png
Saved latency_phases_naive-fib_500.png
Saved latency_phases_focus_naive-fib_500.png
Saved latency_phases_naive-fib-heavy_50.png
Saved latency_phases_focus_naive-fib-heavy_50.png
Saved latency_phases_matrix-mul_500.png
Saved latency_phases_focus_matrix-mul_500.png
Saved latency_phases_matrix-mul-heavy_50.png
Saved latency_phases_focus_matrix-mul-heavy_50.png


In [53]:
print("=== Throughput Statistics (tasks/sec) ===\n")
for task_name in WORKLOADS.keys():
    print(f"Task: {task_name}")
    fig, ax = plt.subplots(figsize=(8, 5))

    for scheduler in SCHEDULERS:
        throughputs = []
        for concurrency in THROUGHPUT_CLIENTS:
            result = throughput_benchmark_results.get(f"{scheduler}_{task_name}_{concurrency}")
            if result:
                tps = result["throughput_tps"]
                throughputs.append(tps)
                print(f"  {scheduler}, concurrency={concurrency}: "
                      f"tps={tps:.2f}, submitted={result['submitted']}, "
                      f"completed={result['completed_before_cutoff']}")
            else:
                throughputs.append(0)
        ax.plot(THROUGHPUT_CLIENTS, throughputs, marker="o", label=scheduler)

    print()
    ax.set_xlabel("Concurrency level (parallel clients)")
    ax.set_ylabel("Throughput (tasks/sec)")
    ax.set_title(f"Throughput Scaling | Task: {task_name}")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(f"benchmark-results/throughput_{task_name}.png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved throughput_{task_name}.png\n")

=== Throughput Statistics (tasks/sec) ===

Task: addition
  roundrobin, concurrency=1: tps=6.30, submitted=191, completed=189
  roundrobin, concurrency=2: tps=14.27, submitted=433, completed=428
  roundrobin, concurrency=4: tps=12.47, submitted=423, completed=374
  roundrobin, concurrency=8: tps=6.47, submitted=448, completed=194
  probabilistic, concurrency=1: tps=6.40, submitted=194, completed=192
  probabilistic, concurrency=2: tps=12.83, submitted=389, completed=385
  probabilistic, concurrency=4: tps=8.23, submitted=365, completed=247
  probabilistic, concurrency=8: tps=7.03, submitted=324, completed=211

Saved throughput_addition.png

Task: naive-fib
  roundrobin, concurrency=1: tps=4.50, submitted=151, completed=135
  roundrobin, concurrency=2: tps=4.47, submitted=150, completed=134
  roundrobin, concurrency=4: tps=2.23, submitted=155, completed=67
  roundrobin, concurrency=8: tps=1.27, submitted=161, completed=38
  probabilistic, concurrency=1: tps=3.07, submitted=111, complete

## Throughput

TPS at each concurrency level for the lightweight workloads. Saved to `benchmark-results/throughput_<workload>.png`.

In [54]:
import csv

os.makedirs("benchmark-results", exist_ok=True)

# Latency stats per phase
with open("benchmark-results/latency_stats.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["scheduler", "task", "rep_count", "phase",
                "n", "mean_ms", "std_ms", "median_ms", "p95_ms", "p99_ms", "iqr_ms", "ci95_low_ms", "ci95_high_ms"])
    for task_name, task_info in WORKLOADS.items():
        for rep_count in task_info.get("rep_counts", REP_COUNTS):
            for scheduler in SCHEDULERS:
                for phase, s in latency_stats.get(f"{scheduler}_{task_name}_{rep_count}", {}).items():
                    if s:
                        w.writerow([scheduler, task_name, rep_count, phase,
                                    s["n"], s["mean"], s["std"], s["median"],
                                    s["p95"], s["p99"], s["iqr"], s["ci95_low"], s["ci95_high"]])

# Energy stats
with open("benchmark-results/energy_stats.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["scheduler", "task", "rep_count", "n", "mean", "std", "median", "p95", "p99", "total"])
    for task_name, task_info in WORKLOADS.items():
        for rep_count in task_info.get("rep_counts", REP_COUNTS):
            for scheduler in SCHEDULERS:
                value = results.get(f"{scheduler}_{task_name}_{rep_count}")
                if not value:
                    continue
                vals = np.array([t["energy_consumed"] for t in value["results"] if t.get("energy_consumed") is not None], dtype=float)
                if len(vals) > 0:
                    w.writerow([scheduler, task_name, rep_count, len(vals),
                                vals.mean(), vals.std(ddof=1), np.median(vals),
                                np.percentile(vals, 95), np.percentile(vals, 99), vals.sum()])

# Throughput stats
with open("benchmark-results/throughput_stats.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["scheduler", "task", "concurrency", "throughput_tps", "submitted", "completed"])
    for task_name, task_info in WORKLOADS.items():
        if task_info.get("skip_throughput"):
            continue
        for scheduler in SCHEDULERS:
            for concurrency in THROUGHPUT_CLIENTS:
                result = throughput_benchmark_results.get(f"{scheduler}_{task_name}_{concurrency}")
                if result:
                    w.writerow([scheduler, task_name, concurrency,
                                result["throughput_tps"], result["submitted"], result["completed_before_cutoff"]])

print("Exported:")
print("  benchmark-results/latency_stats.csv")
print("  benchmark-results/energy_stats.csv")
print("  benchmark-results/throughput_stats.csv")

Exported:
  benchmark-results/latency_stats.csv
  benchmark-results/energy_stats.csv
  benchmark-results/throughput_stats.csv
